In [116]:
import boto3
import pandas as pd
import numpy as np
import io
import torch
from sklearn.preprocessing import MinMaxScaler


In [117]:
s3_client = boto3.client(
    's3', 
    endpoint_url='http://localstack:4566',
    # Como é LocalStack, podemos passar credenciais falsas qualquer
    aws_access_key_id='test',
    aws_secret_access_key='test',
    region_name='us-east-1'
)

bucket_name = 'data-extract'
archive_path = 'landing/01-07-2026/raw_PETR4.csv'

try:
    print(f"Baixando arquivo {archive_path} do bucket {bucket_name}...")
    # Recuperamos o arquivo do S3
    response = s3_client.get_object(Bucket=bucket_name, Key=archive_path)
    
    # Lemos o conteúdo do arquivo CSV em um DataFrame do Pandas
    bytes_data = response['Body'].read()
    df = pd.read_csv(io.BytesIO(bytes_data), header=[0, 1], index_col=0)
    # Ajustamos o DataFrame para remover o nível de cabeçalho extra
    df.columns = df.columns.droplevel(1)
    # Ajustamos os nomes das colunas e do índice
    df.columns.name = None
    df.index.name = 'Date'
    df = df.reset_index()
    
    print("Arquivo baixado com sucesso!")
    print("Exibindo as primeiras linhas do DataFrame:")
    print(df.head(1))
except Exception as e:
    print(f"Erro ao baixar o arquivo: {e}")

Baixando arquivo landing/01-07-2026/raw_PETR4.csv do bucket data-extract...
Arquivo baixado com sucesso!
Exibindo as primeiras linhas do DataFrame:
         Date     Close      High       Low      Open    Volume
0  2016-06-30  2.433883  2.451969  2.387375  2.433883  43741300


# Tratamento do dataset
- Como nosso dataset está totalmente limpo, irei aplicar engenharia de feature

In [118]:
df_ef = df.copy()

# Cálculo das médias móveis simples (SMA) para diferentes janelas de tempo em 5, 10, 20 e 30 dias
df_ef["SMA_5"] = df_ef['Close'].rolling(window=5).mean()
df_ef["SMA_15"] = df_ef['Close'].rolling(window=15).mean()
df_ef["SMA_30"] = df_ef['Close'].rolling(window=30).mean()

# Cálculo das médias móveis exponenciais (EMA) para diferentes janelas de tempo em 5, 10, 20 e 30 dias
df_ef["EMA_5"] = df_ef['Close'].ewm(span=5, adjust=False).mean()
df_ef["EMA_15"] = df_ef['Close'].ewm(span=15, adjust=False).mean()
df_ef["EMA_30"] = df_ef['Close'].ewm(span=30, adjust=False).mean()

# Cálculo da variação percentual diária do preço de fechamento e do volume negociado
df_ef['Variation_pct'] = df_ef['Close'].pct_change()
df_ef['Vol_variation_pct'] = df_ef['Volume'].pct_change()
df_ef['Vol_variation_pct_10'] = df_ef['Volume'].pct_change(periods=10)
df_ef.head(30)

,Date,Close,High,Low,Open,Volume,SMA_5,SMA_15,SMA_30,EMA_5,EMA_15,EMA_30,Variation_pct,Vol_variation_pct,Vol_variation_pct_10
0,2016-06-30,2.433883,2.451969,2.387375,2.433883,43741300,NaN,NaN,NaN,2.433883,2.433883,2.433883,NaN,NaN,NaN
1,2016-07-01,2.537231,2.544983,2.420963,2.449384,49290800,NaN,NaN,NaN,2.468332,2.446801,2.440550,0.042462,0.126871,NaN
2,2016-07-04,2.550150,2.596658,2.529481,2.570820,24505100,NaN,NaN,NaN,2.495605,2.459720,2.447621,0.005092,-0.502846,NaN
3,2016-07-05,2.400294,2.498476,2.384792,2.493309,69691200,NaN,NaN,NaN,2.463835,2.452292,2.444568,-0.058764,1.843947,NaN
4,2016-07-06,2.454552,2.459720,2.312446,2.348619,68781500,2.475222,NaN,NaN,2.460741,2.452574,2.445212,0.022605,-0.013053,NaN
5,2016-07-07,2.470055,2.578572,2.459720,2.495892,81610300,2.482457,NaN,NaN,2.463845,2.454759,2.446815,0.006316,0.186515,NaN
6,2016-07-08,2.542400,2.568237,2.498476,2.544983,42647600,2.483490,NaN,NaN,2.490030,2.465714,2.452982,0.029289,-0.477424,NaN
7,2016-07-11,2.676754,2.679337,2.578571,2.583739,66717300,2.508811,NaN,NaN,2.552271,2.492094,2.467419,0.052845,0.564386,NaN
8,2016-07-12,2.751683,2.790439,2.733596,2.741348,58174200,2.579089,NaN,NaN,2.618742,2.524543,2.485758,0.027992,-0.128049,NaN
9,2016-07-13,2.746515,2.746515,2.637998,2.705175,64252100,2.637481,NaN,NaN,2.661333,2.552289,2.502581,-0.001878,0.104478,NaN


In [119]:
df_ef['Date'] = pd.to_datetime(df_ef['Date'])
# Adicionando colunas para o dia da semana e o mês do ano para o modelo entender padrões sazonais
df_ef["Day_of_week_num"] = df_ef['Date'].dt.weekday
df_ef["Month_num"] = df_ef['Date'].dt.month
df_ef.head(30)

,Date,Close,High,Low,Open,Volume,SMA_5,SMA_15,SMA_30,EMA_5,EMA_15,EMA_30,Variation_pct,Vol_variation_pct,Vol_variation_pct_10,Day_of_week_num,Month_num
0,2016-06-30,2.433883,2.451969,2.387375,2.433883,43741300,NaN,NaN,NaN,2.433883,2.433883,2.433883,NaN,NaN,NaN,3,6
1,2016-07-01,2.537231,2.544983,2.420963,2.449384,49290800,NaN,NaN,NaN,2.468332,2.446801,2.440550,0.042462,0.126871,NaN,4,7
2,2016-07-04,2.550150,2.596658,2.529481,2.570820,24505100,NaN,NaN,NaN,2.495605,2.459720,2.447621,0.005092,-0.502846,NaN,0,7
3,2016-07-05,2.400294,2.498476,2.384792,2.493309,69691200,NaN,NaN,NaN,2.463835,2.452292,2.444568,-0.058764,1.843947,NaN,1,7
4,2016-07-06,2.454552,2.459720,2.312446,2.348619,68781500,2.475222,NaN,NaN,2.460741,2.452574,2.445212,0.022605,-0.013053,NaN,2,7
5,2016-07-07,2.470055,2.578572,2.459720,2.495892,81610300,2.482457,NaN,NaN,2.463845,2.454759,2.446815,0.006316,0.186515,NaN,3,7
6,2016-07-08,2.542400,2.568237,2.498476,2.544983,42647600,2.483490,NaN,NaN,2.490030,2.465714,2.452982,0.029289,-0.477424,NaN,4,7
7,2016-07-11,2.676754,2.679337,2.578571,2.583739,66717300,2.508811,NaN,NaN,2.552271,2.492094,2.467419,0.052845,0.564386,NaN,0,7
8,2016-07-12,2.751683,2.790439,2.733596,2.741348,58174200,2.579089,NaN,NaN,2.618742,2.524543,2.485758,0.027992,-0.128049,NaN,1,7
9,2016-07-13,2.746515,2.746515,2.637998,2.705175,64252100,2.637481,NaN,NaN,2.661333,2.552289,2.502581,-0.001878,0.104478,NaN,2,7


- Vamos adicionar a métrica de RSI ao dataset

In [120]:
periodo = 14
# Calculando o ganho e a perda entre o dia atual e o dia anterior
df_ef['Delta'] = df_ef['Close'].diff()
# Separamos os ganhos e perdas em colunas distintas, substituindo valores negativos por 0 na coluna de ganhos e valores positivos por 0 na coluna de perdas
df_ef['Ganho'] = df_ef['Delta'].where(df_ef['Delta'] > 0, 0)
df_ef['Perda'] = df_ef['Delta'].where(df_ef['Delta'] < 0, 0).abs()
# Calculamos a média movel exponencial (EMA) dos ganhos e perdas para suavizar os valores e obter uma visão mais clara da tendência de alta ou baixa do ativo
df_ef['Media_ganho'] = df_ef['Ganho'].ewm(alpha=1/periodo, adjust=False).mean()
df_ef['Media_perda'] = df_ef['Perda'].ewm(alpha=1/periodo, adjust=False).mean()

# Cálculo do Índice de Força Relativa (RSI) com base nas médias móveis exponenciais dos ganhos e perdas
df_ef['RS'] = df_ef['Media_ganho'] / df_ef['Media_perda']
df_ef['RSI'] = 100 - (100 / (1 + df_ef['RS']))

# Removendo colunas desnecessárias do DataFrame
df_ef = df_ef.drop(columns=['Delta', 'Ganho', 'Perda', 'Media_ganho', 'Media_perda', 'RS'])
# Retirada de valores nulos do DataFrame resultante
df_ef.dropna(inplace=True)

df_ef.head(30)

,Date,Close,High,Low,Open,Volume,SMA_5,SMA_15,SMA_30,EMA_5,EMA_15,EMA_30,Variation_pct,Vol_variation_pct,Vol_variation_pct_10,Day_of_week_num,Month_num,RSI
29,2016-08-10,2.986802,3.092736,2.986802,3.087568,41638400,3.049329,3.032276,2.860027,3.030814,2.993343,2.884163,-0.026936,0.239378,-0.198523,2,8,55.523505
30,2016-08-11,3.126324,3.126324,2.997137,3.004888,55881500,3.053979,3.036582,2.883108,3.062651,3.009965,2.899787,0.046713,0.342066,0.374256,3,8,62.574738
31,2016-08-12,3.100487,3.214172,3.095320,3.113406,71281300,3.072066,3.037616,2.901884,3.075263,3.021281,2.912735,-0.008264,0.275580,0.173548,4,8,60.656972
32,2016-08-15,3.180584,3.196086,3.118574,3.149579,53568000,3.092736,3.042439,2.922898,3.110370,3.041193,2.930016,0.025834,-0.248499,0.029910,0,8,64.308864
33,2016-08-16,3.227090,3.245176,3.136659,3.180583,48218700,3.124257,3.052946,2.950458,3.149276,3.064430,2.949182,0.014622,-0.099860,-0.018641,1,8,66.266752
34,2016-08-17,3.296850,3.296850,3.183166,3.221922,46809100,3.186267,3.074477,2.978534,3.198468,3.093483,2.971612,0.021617,-0.029233,-0.159553,2,8,69.012730
35,2016-08-18,3.327857,3.366613,3.302019,3.307187,44941100,3.226574,3.099281,3.007128,3.241597,3.122780,2.994595,0.009405,-0.039907,-0.288991,3,8,70.174848
36,2016-08-19,3.304602,3.338190,3.281348,3.312353,33360000,3.267396,3.115128,3.032535,3.262599,3.145507,3.014596,-0.006988,-0.257695,-0.384796,4,8,68.111665
37,2016-08-22,3.190918,3.263263,3.177999,3.263263,43258900,3.269463,3.134075,3.049673,3.238705,3.151184,3.025971,-0.034402,0.296730,0.041079,0,8,58.982285
38,2016-08-23,3.273598,3.322689,3.209004,3.224507,61048000,3.278765,3.156640,3.067071,3.250336,3.166485,3.041947,0.025911,0.411224,0.817110,1,8,62.879212


In [121]:
df_ef.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2462 entries, 29 to 2490
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Date                  2462 non-null   datetime64[ns]
 1   Close                 2462 non-null   float64       
 2   High                  2462 non-null   float64       
 3   Low                   2462 non-null   float64       
 4   Open                  2462 non-null   float64       
 5   Volume                2462 non-null   int64         
 6   SMA_5                 2462 non-null   float64       
 7   SMA_15                2462 non-null   float64       
 8   SMA_30                2462 non-null   float64       
 9   EMA_5                 2462 non-null   float64       
 10  EMA_15                2462 non-null   float64       
 11  EMA_30                2462 non-null   float64       
 12  Variation_pct         2462 non-null   float64       
 13  Vol_variation_pct     

In [122]:
df_ef.describe()

,Date,Close,High,Low,Open,Volume,SMA_5,SMA_15,SMA_30,EMA_5,EMA_15,EMA_30,Variation_pct,Vol_variation_pct,Vol_variation_pct_10,Day_of_week_num,Month_num,RSI
count,2462,2462.000000,2462.000000,2462.000000,2462.000000,2.462000e+03,2462.000000,2462.000000,2462.000000,2462.000000,2462.000000,2462.000000,2462.000000,2462.000000,2462.000000,2462.000000,2462.000000,2462.000000
mean,2021-07-19 02:10:25.832656384,14.507764,14.688275,14.326781,14.507776,5.731829e+07,14.479367,14.406779,14.291924,14.479216,14.404924,14.285972,0.001362,inf,inf,1.998375,6.499594,54.252505
min,2016-08-10 00:00:00,2.986802,3.092736,2.986802,3.004888,0.000000e+00,3.049329,3.032276,2.860027,3.030814,2.993343,2.884163,-0.296978,-1.000000,-1.000000,0.000000,1.000000,16.966736
25%,2019-01-28 06:00:00,5.798445,5.884223,5.710678,5.813398,3.547318e+07,5.797228,5.759714,5.727603,5.792549,5.763252,5.720273,-0.010415,-0.232293,-0.294564,1.000000,4.000000,45.883150
50%,2021-07-22 12:00:00,8.279569,8.397094,8.167609,8.291711,4.992425e+07,8.265964,8.195806,8.179417,8.251492,8.208371,8.145409,0.001280,-0.015853,-0.017931,2.000000,6.500000,54.340688
75%,2024-01-10 18:00:00,26.101257,26.443049,25.799240,26.095929,6.930145e+07,25.957001,26.096047,25.734812,26.083197,26.392101,26.021434,0.013712,0.291020,0.403730,3.000000,9.000000,62.609798
max,2026-06-30 00:00:00,48.523716,49.157930,48.110663,48.556670,4.902304e+08,47.906105,46.899001,46.782045,47.832550,47.108034,46.039362,0.222222,inf,inf,4.000000,12.000000,88.396233
std,NaN,11.293236,11.398614,11.180101,11.285214,3.358626e+07,11.273818,11.221680,11.128476,11.271037,11.209253,11.098343,0.025842,NaN,NaN,1.411336,3.437578,12.251503


In [123]:
df_clean = df_ef.copy()
df_clean = df_clean.drop(columns=['High', 'Low', 'Open'])
df_clean = df_clean[['Date', 'Close', 'Variation_pct', 'SMA_5', 'SMA_15', 'SMA_30', 'EMA_5', 'EMA_15', 'EMA_30', 'RSI', 'Volume', 'Vol_variation_pct', 'Vol_variation_pct_10', 'Day_of_week_num', 'Month_num',]]
df_clean.head(30)

,Date,Close,Variation_pct,SMA_5,SMA_15,SMA_30,EMA_5,EMA_15,EMA_30,RSI,Volume,Vol_variation_pct,Vol_variation_pct_10,Day_of_week_num,Month_num
29,2016-08-10,2.986802,-0.026936,3.049329,3.032276,2.860027,3.030814,2.993343,2.884163,55.523505,41638400,0.239378,-0.198523,2,8
30,2016-08-11,3.126324,0.046713,3.053979,3.036582,2.883108,3.062651,3.009965,2.899787,62.574738,55881500,0.342066,0.374256,3,8
31,2016-08-12,3.100487,-0.008264,3.072066,3.037616,2.901884,3.075263,3.021281,2.912735,60.656972,71281300,0.275580,0.173548,4,8
32,2016-08-15,3.180584,0.025834,3.092736,3.042439,2.922898,3.110370,3.041193,2.930016,64.308864,53568000,-0.248499,0.029910,0,8
33,2016-08-16,3.227090,0.014622,3.124257,3.052946,2.950458,3.149276,3.064430,2.949182,66.266752,48218700,-0.099860,-0.018641,1,8
34,2016-08-17,3.296850,0.021617,3.186267,3.074477,2.978534,3.198468,3.093483,2.971612,69.012730,46809100,-0.029233,-0.159553,2,8
35,2016-08-18,3.327857,0.009405,3.226574,3.099281,3.007128,3.241597,3.122780,2.994595,70.174848,44941100,-0.039907,-0.288991,3,8
36,2016-08-19,3.304602,-0.006988,3.267396,3.115128,3.032535,3.262599,3.145507,3.014596,68.111665,33360000,-0.257695,-0.384796,4,8
37,2016-08-22,3.190918,-0.034402,3.269463,3.134075,3.049673,3.238705,3.151184,3.025971,58.982285,43258900,0.296730,0.041079,0,8
38,2016-08-23,3.273598,0.025911,3.278765,3.156640,3.067071,3.250336,3.166485,3.041947,62.879212,61048000,0.411224,0.817110,1,8


## Vamos efetuar a normalização dos dados
- Vou utilizar o MimMaxScaler com dados sendo criados entre -1 a 1

In [124]:
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
df_clean = df_clean.dropna()

target = df_clean['Close']
features = df_clean.drop(columns=['Date'])


scaler_target = MinMaxScaler(feature_range=(0, 1))
target_normalized = scaler_target.fit_transform(target.values.reshape(-1, 1))

scaler_all = MinMaxScaler(feature_range=(0, 1))
data_normalized = scaler_all.fit_transform(features)



In [ ]:
def sliding_window(data, window_size: int) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Creates sliding windows from the data for time series forecasting.
    Parameters:
    - data: Numpy array of shape (num_samples, num_features).
    - window_size: Size of the sliding window.
    Returns:
    - Tuple of tensors (X, y) where X is the input windows and y is the target values.
    """
    
    X = []
    y = []
    
    # Loop que percorre os dados criando fotos do passado
    for i in range(len(data) - window_size):
        window = data[i:(i + window_size)]
        
        target = data[i + window_size]

        target_value = target[0]
        
        X.append(window)
        y.append(target_value)
        
    return torch.tensor(np.array(X).astype(np.float32)), torch.tensor(np.array(y).astype(np.float32))

X_tensor, y_tensor = sliding_window(data_normalized, window_size=15)

# Dividindo em treino e teste (80% treino, 20% teste)
train_size = int(len(X_tensor) * 0.8)
X_train = X_tensor[:train_size]
y_train = y_tensor[:train_size]
X_test = X_tensor[train_size:]
y_test = y_tensor[train_size:]